# FSAI, Inc. — DIA (Disaster Intelligence Atlas)
## Phase 2: Model Development, Training & Iterative Tuning
**Input:** the Phase 1 outputs (`FSAI_DIA_Phase1_outputs.zip`). **Study area:** 14 LGUs of Zambales + Olongapo City.

**Design (option A).** The data has two very different sample sizes, and the models are split to match:

| Level | Rows | What can be validated |
|---|---|---|
| **Daily rainfall** (3 distinct NASA POWER cells × 16,742 days) | ~50,000 | Real predictive models with forward-chaining time-series CV and a 2021–2026 hold-out → **main predictive section** |
| **LGU** (14 municipalities/city) | 14 | Descriptive structure only; any classifier/regressor here is **exploratory** and is checked with leave-one-out CV + permutation tests |

**Fixes carried over from the Phase 1 quality assessment:**
1. Overlapping hazard classes in the NOAH 100-yr layer are resolved (each piece of land keeps only its *highest* class).
2. NOAH's 5- and 25-yr maps do not cover Candelaria, Masinloc, Palauig and Santa Cruz. Those LGUs are flagged `mapped_5_25yr = False`, and cross-scenario comparisons use only the mapped LGUs.
3. Rainfall statistics use the 3 *distinct* POWER cells, not 14 copies.

Every model run is logged (algorithm, hyper-parameters, validation scheme, metrics, what changed) to `experiment_log.csv` / `.md`.

**How to run:** In Colab, run all cells. The first cell asks you to upload `FSAI_DIA_Phase1_outputs.zip` if it isn't already present. Runtime is about 5–10 minutes.

## 0. Setup

In [1]:
import importlib, subprocess, sys
for pkg, mod in [("geopandas","geopandas"), ("pyogrio","pyogrio"), ("mlxtend","mlxtend"), ("networkx","networkx"),
                 ("scikit-learn","sklearn"), ("joblib","joblib")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [2]:
import os, json, zipfile, shutil, warnings, time, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt, seaborn as sns
import networkx as nx, joblib
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, adjusted_rand_score, roc_auc_score, average_precision_score,
                             brier_score_loss, precision_recall_curve, precision_score, recall_score, f1_score,
                             accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict, permutation_test_score, StratifiedKFold
from sklearn.inspection import permutation_importance
from mlxtend.frequent_patterns import apriori, association_rules
warnings.filterwarnings("ignore")
SEED = 42; rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": .6, "font.size": 10,
                     "axes.titleweight": "bold", "axes.titlesize": 11})
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
SEQ3 = ["#86b6ef", "#2a78d6", "#104281"]

### Configuration and data loading

In [3]:
PHASE1_ZIP = Path("FSAI_DIA_Phase1_outputs.zip")
DATA = Path(os.environ.get("DIA_PHASE1_DIR", "phase1_outputs"))
OUT = Path("phase2_outputs"); FIG = OUT/"figures"; FIG.mkdir(parents=True, exist_ok=True)

if not (DATA/"run_summary.json").exists():
    if not PHASE1_ZIP.exists():
        try:
            from google.colab import files
            print("Upload FSAI_DIA_Phase1_outputs.zip"); up = files.upload()
            PHASE1_ZIP = Path(next(iter(up)))
        except ImportError:
            raise FileNotFoundError("Put FSAI_DIA_Phase1_outputs.zip next to this notebook.")
    with zipfile.ZipFile(PHASE1_ZIP) as z: z.extractall(DATA)

BASELINE_YEARS = (1981, 2010)   # WMO-style climatological normal: defines the 'extreme day' threshold
HOLDOUT_START  = 2021           # 2021-01-01 .. latest day = untouched test period
CV_TEST_BLOCKS = [(2001, 2005), (2006, 2010), (2011, 2015), (2016, 2020)]  # forward-chaining folds
WET_SEASON = [5, 6, 7, 8, 9, 10]

phase1 = json.loads((DATA/"run_summary.json").read_text())
lgu_p1 = pd.read_csv(DATA/"clean/lgu_features.csv")
rain = pd.read_csv(DATA/"clean/rainfall_daily_clean.csv", parse_dates=["date"])
hz = gpd.read_file(DATA/"clean/dia_hazard_by_lgu.gpkg", engine="pyogrio")
bounds = gpd.read_file(DATA/"clean/dia_lgu_boundaries.geojson")
CELL_GROUPS = phase1["power_cell_groups"]; REPS = [g[0] for g in CELL_GROUPS]
print("LGUs:", len(lgu_p1), "| hazard features:", len(hz), "| daily rows:", len(rain), "| distinct POWER cells:", REPS)

LGUs: 14 | hazard features: 246 | daily rows: 234388 | distinct POWER cells: ['Botolan', 'Candelaria', 'Santa Cruz']


In [4]:
LOG = []
def log_run(run_id, section, task, algorithm, features, params, validation, metrics, change, reliability="ok"):
    # One row per experimental run. reliability = 'ok' | 'exploratory (n=14)' | 'unreliable: <reason>'
    row = {"run_id": run_id, "section": section, "task": task, "algorithm": algorithm,
           "features": features if isinstance(features, str) else ", ".join(features),
           "hyperparameters": json.dumps(params, default=str), "validation": validation,
           **{k: (round(float(v), 4) if isinstance(v, (int, float, np.floating, np.integer)) and v is not None else v)
              for k, v in metrics.items()},
           "change_vs_previous": change, "reliability": reliability}
    LOG.append(row); return row

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/f"{name}.png", bbox_inches="tight"); plt.show(); plt.close()

## 1. Data preparation (Phase 1 fixes)
### 1.1 Resolve overlapping hazard classes
Within one scenario, a piece of land should carry exactly one hazard level. Here the highest level wins: High stays as is, Medium becomes Medium minus High, and Low becomes Low minus (High ∪ Medium). Phase 1 found 6.04 km² of double counting in San Narciso (100-yr).

In [5]:
t0 = time.time(); fixed = []
for (n, k, s), grp in hz.groupby(["lgu_name", "hazard_type", "scenario"]):
    geo = {int(r.hazard_level): r.geometry for r in grp.itertuples()}
    higher = None
    for lvl in (3, 2, 1):
        if lvl not in geo: continue
        g = geo[lvl] if higher is None else geo[lvl].difference(higher)
        fixed.append({"lgu_name": n, "hazard_type": k, "scenario": s, "hazard_level": lvl,
                      "area_km2_raw": geo[lvl].area/1e6, "area_km2": g.area/1e6,
                      "lgu_area_km2": grp.lgu_area_km2.iloc[0]})
        higher = geo[lvl] if higher is None else higher.union(geo[lvl])
hzf = pd.DataFrame(fixed); hzf["share_pct"] = hzf.area_km2/hzf.lgu_area_km2*100
ovl = (hzf.area_km2_raw - hzf.area_km2).groupby([hzf.hazard_type, hzf.scenario]).sum().round(3)
print(f"done in {time.time()-t0:.0f}s; double-counted area removed (km²):"); print(ovl)
OVERLAP_REMOVED = ovl.to_dict()

done in 79s; double-counted area removed (km²):
hazard_type  scenario
flood        100yr       6.796
             25yr       -0.000
             5yr         0.000
storm_surge  SSA1        0.004
             SSA2        0.047
             SSA3        0.119
             SSA4        0.213
dtype: float64


### 1.2 Corrected LGU feature table + coverage flags

In [6]:
LVL = {1: "low", 2: "medium", 3: "high"}
w = hzf.assign(col=lambda d: np.where(d.hazard_type == "flood", "flood_", "ss_") + d.scenario.str.lower() + "_" + d.hazard_level.map(LVL) + "_pct") \
       .pivot_table(index="lgu_name", columns="col", values="share_pct", aggfunc="sum").fillna(0.0)
for s in ["5yr", "25yr", "100yr"]:
    w[f"flood_{s}_any_pct"] = w[[c for c in w if c.startswith(f"flood_{s}_")]].sum(axis=1)
for a in ["ssa1", "ssa2", "ssa3", "ssa4"]:
    w[f"ss_{a}_any_pct"] = w[[c for c in w if c.startswith(f"ss_{a}_")]].sum(axis=1)

# Rainfall climatology per distinct cell, mapped back to LGUs
def cell_stats(d):
    d = d.set_index("date").PRECTOTCORR; y = d.groupby(d.index.year); full = y.count() >= 360
    return pd.Series({"rain_mean_annual_mm": y.sum()[full].mean(), "rain_mean_annual_max1d_mm": y.max()[full].mean(),
                      "rain_p95_wetday_mm": d[d >= 1].quantile(.95),
                      "rain_wetseason_share_pct": d[d.index.month.isin(WET_SEASON)].sum()/d.sum()*100})
cs = rain[rain.lgu_name.isin(REPS)].groupby("lgu_name").apply(cell_stats)
rep_of = {n: g[0] for g in CELL_GROUPS for n in g}

L = lgu_p1[["lgu_name", "centroid_lat", "centroid_lon", "lgu_area_km2"]].set_index("lgu_name").join(w)
L["rain_cell"] = L.index.map(rep_of)
L = L.join(cs, on="rain_cell")
L["mapped_5_25yr"] = L.flood_5yr_any_pct > 0
L["high_growth_pp"] = np.where(L.mapped_5_25yr, L.flood_100yr_high_pct - L.flood_5yr_high_pct, np.nan)
L.to_csv(OUT/"lgu_features_corrected.csv")
print("Not mapped at 5/25-yr:", L.index[~L.mapped_5_25yr].tolist())
L[["rain_cell", "mapped_5_25yr", "flood_5yr_high_pct", "flood_100yr_high_pct", "flood_100yr_any_pct", "ss_ssa4_any_pct",
   "rain_mean_annual_mm", "rain_mean_annual_max1d_mm"]].round(2)

Not mapped at 5/25-yr: ['Candelaria', 'Masinloc', 'Palauig', 'Santa Cruz']


,rain_cell,mapped_5_25yr,flood_5yr_high_pct,flood_100yr_high_pct,flood_100yr_any_pct,ss_ssa4_any_pct,rain_mean_annual_mm,rain_mean_annual_max1d_mm
lgu_name,,,,,,,,
Botolan,Botolan,True,8.67,15.97,27.43,2.73,2648.88,132.04
Cabangan,Botolan,True,2.96,8.83,17.98,7.86,2648.88,132.04
Candelaria,Candelaria,False,0.00,5.72,14.04,6.22,2396.56,107.94
Castillejos,Botolan,True,7.44,13.14,36.89,0.00,2648.88,132.04
Iba,Candelaria,True,7.06,15.51,30.72,18.03,2396.56,107.94
Masinloc,Candelaria,False,0.00,7.83,16.28,7.55,2396.56,107.94
Olongapo,Botolan,True,0.17,7.68,16.94,10.22,2648.88,132.04
Palauig,Candelaria,False,0.00,6.11,15.11,8.33,2396.56,107.94
San Antonio,Botolan,True,2.82,6.14,26.70,8.89,2648.88,132.04


## 2. Descriptive models
### 2.1 Correlation matrix (Spearman)
Spearman correlation is used instead of Pearson: n = 14, the shares are skewed (San Narciso is an outlier), and the rainfall columns take only 3 distinct values.

In [7]:
CORR_COLS = ["flood_100yr_high_pct", "flood_100yr_medium_pct", "flood_100yr_low_pct", "flood_100yr_any_pct",
             "ss_ssa1_any_pct", "ss_ssa4_any_pct", "rain_mean_annual_mm", "rain_mean_annual_max1d_mm",
             "lgu_area_km2", "centroid_lat"]
C = L[CORR_COLS].corr(method="spearman")
plt.figure(figsize=(8.5, 7)); sns.heatmap(C, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
                                          cbar_kws={"shrink": .6}, annot_kws={"size": 7})
plt.gca().grid(False); plt.title("Spearman correlation, LGU features (n = 14)"); savefig("p2_01_lgu_spearman")
strong = [(a, b, round(C.loc[a, b], 2)) for i, a in enumerate(CORR_COLS) for b in CORR_COLS[i+1:] if abs(C.loc[a, b]) >= .7]
print("|rho| ≥ 0.7:", strong)
log_run("D1", "descriptive", "LGU feature correlation", "Spearman correlation matrix", CORR_COLS, {}, "full data (n=14)",
        {"pairs_abs_rho_ge_0.7": len(strong)}, "first run", "exploratory (n=14)")

|rho| ≥ 0.7: [('flood_100yr_high_pct', 'flood_100yr_any_pct', np.float64(0.83)), ('flood_100yr_medium_pct', 'flood_100yr_low_pct', np.float64(0.85)), ('flood_100yr_medium_pct', 'flood_100yr_any_pct', np.float64(0.89)), ('flood_100yr_low_pct', 'flood_100yr_any_pct', np.float64(0.92)), ('rain_mean_annual_mm', 'rain_mean_annual_max1d_mm', np.float64(1.0)), ('rain_mean_annual_mm', 'centroid_lat', np.float64(-0.79)), ('rain_mean_annual_max1d_mm', 'centroid_lat', np.float64(-0.79))]


{'run_id': 'D1',
 'section': 'descriptive',
 'task': 'LGU feature correlation',
 'algorithm': 'Spearman correlation matrix',
 'features': 'flood_100yr_high_pct, flood_100yr_medium_pct, flood_100yr_low_pct, flood_100yr_any_pct, ss_ssa1_any_pct, ss_ssa4_any_pct, rain_mean_annual_mm, rain_mean_annual_max1d_mm, lgu_area_km2, centroid_lat',
 'hyperparameters': '{}',
 'validation': 'full data (n=14)',
 'pairs_abs_rho_ge_0.7': 7.0,
 'change_vs_previous': 'first run',
 'reliability': 'exploratory (n=14)'}

### 2.2 k-Means clustering of LGUs
Three feature sets were tried, iterating on what the earlier run revealed. k = 2–6 is scored by silhouette (higher is better) and inertia (elbow). The chosen solution's stability is then checked with 300 bootstrap resamples (adjusted Rand index vs. the full-data clustering).

⚠️ **Small-N flag:** silhouette on 14 points moves a lot when one LGU changes cluster. The bootstrap stability check is what decides whether a clustering can be trusted.

In [8]:
KM_SETS = {
  "KM-A hazard+rain": (["flood_100yr_high_pct", "flood_100yr_medium_pct", "flood_100yr_low_pct", "ss_ssa4_any_pct",
                        "rain_mean_annual_mm", "rain_mean_annual_max1d_mm"], False,
                       "first run: hazard + rainfall"),
  "KM-B hazard only": (["flood_100yr_high_pct", "flood_100yr_medium_pct", "flood_100yr_low_pct", "ss_ssa4_any_pct"], False,
                       "dropped rainfall: KM-A clusters tracked the 3 POWER cells, not hazard"),
  "KM-C hazard log1p": (["flood_100yr_high_pct", "flood_100yr_medium_pct", "flood_100yr_low_pct", "ss_ssa4_any_pct"], True,
                        "log1p(shares) to damp the San Narciso outlier"),
}
km_res = {}; rid = 0
for name, (cols, logt, why) in KM_SETS.items():
    X = L[cols].to_numpy(float); X = np.log1p(X) if logt else X
    Xs = StandardScaler().fit_transform(X)
    for k in range(2, 7):
        km = KMeans(n_clusters=k, n_init=50, random_state=SEED).fit(Xs)
        sil = silhouette_score(Xs, km.labels_); rid += 1
        km_res[(name, k)] = {"sil": sil, "inertia": km.inertia_, "labels": km.labels_, "Xs": Xs, "cols": cols, "log": logt,
                             "sizes": np.bincount(km.labels_).tolist()}
        ari_rain = adjusted_rand_score(L.rain_cell, km.labels_)
        log_run(f"KM{rid:02d}", "descriptive", "LGU clustering", "k-Means", cols, {"k": k, "n_init": 50, "log1p": logt},
                "full data (n=14)", {"silhouette": sil, "inertia": km.inertia_, "ARI_vs_rain_cell": ari_rain,
                                     "min_cluster_size": min(np.bincount(km.labels_))},
                why if k == 2 else f"k={k}", "exploratory (n=14)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for i, name in enumerate(KM_SETS):
    ks = range(2, 7)
    ax[0].plot(ks, [km_res[(name, k)]["sil"] for k in ks], marker="o", lw=2, color=CAT[i], label=name)
    ax[1].plot(ks, [km_res[(name, k)]["inertia"] for k in ks], marker="o", lw=2, color=CAT[i], label=name)
ax[0].set_title("Silhouette by k"); ax[1].set_title("Inertia (elbow) by k")
for a in ax: a.set_xlabel("k"); a.legend(frameon=False, fontsize=8)
savefig("p2_02_kmeans_selection")
pd.DataFrame([{"set": n, "k": k, "silhouette": round(v["sil"], 3), "sizes": v["sizes"]} for (n, k), v in km_res.items()])

,set,k,silhouette,sizes
0,KM-A hazard+rain,2,0.398,"[4, 10]"
1,KM-A hazard+rain,3,0.421,"[9, 4, 1]"
2,KM-A hazard+rain,4,0.353,"[6, 4, 3, 1]"
3,KM-A hazard+rain,5,0.396,"[6, 1, 3, 3, 1]"
4,KM-A hazard+rain,6,0.349,"[4, 1, 1, 3, 2, 3]"
5,KM-B hazard only,2,0.438,"[11, 3]"
6,KM-B hazard only,3,0.404,"[3, 3, 8]"
7,KM-B hazard only,4,0.439,"[2, 8, 3, 1]"
8,KM-B hazard only,5,0.399,"[8, 2, 2, 1, 1]"
9,KM-B hazard only,6,0.353,"[2, 8, 1, 1, 1, 1]"


In [9]:
# Choose: best silhouette among sets that do NOT just reproduce the rainfall grid (ARI_vs_rain_cell < 0.8)
# and have no singleton cluster unless k=2.
cands = [(n, k, v) for (n, k), v in km_res.items()
         if adjusted_rand_score(L.rain_cell, v["labels"]) < .8 and min(v["sizes"]) >= 2]
if not cands: cands = [(n, k, v) for (n, k), v in km_res.items() if min(v["sizes"]) >= 2]
best_name, best_k, best = max(cands, key=lambda t: t[2]["sil"])
print("Chosen:", best_name, "k =", best_k, "silhouette =", round(best["sil"], 3), "sizes =", best["sizes"])

# Bootstrap stability: refit on resamples, compare labels of the resampled LGUs with the full-data labels
aris = []
for b in range(300):
    idx = rng.choice(len(L), len(L), replace=True)
    if len(np.unique(idx)) < best_k + 1: continue
    kmb = KMeans(n_clusters=best_k, n_init=20, random_state=b).fit(best["Xs"][idx])
    aris.append(adjusted_rand_score(best["labels"][idx], kmb.labels_))
stab = float(np.mean(aris)); print(f"bootstrap ARI mean = {stab:.3f} (≥0.75 stable, 0.6–0.75 fair, <0.6 unstable)")
log_run("KM-FINAL", "descriptive", "LGU clustering (chosen)", "k-Means", best["cols"],
        {"k": best_k, "log1p": best["log"], "set": best_name}, "300 bootstrap resamples",
        {"silhouette": best["sil"], "bootstrap_ARI_mean": stab, "bootstrap_ARI_p05": np.percentile(aris, 5)},
        "selected by silhouette among non-degenerate solutions", "exploratory (n=14)" if stab >= .6 else "unreliable: unstable clusters")

L["cluster"] = best["labels"]
order = L.groupby("cluster").flood_100yr_any_pct.mean().sort_values().index
names = {c: f"C{i+1}" for i, c in enumerate(order)}; L["cluster"] = L.cluster.map(names)
prof = L.groupby("cluster")[best["cols"] + ["flood_100yr_any_pct"]].mean().round(2)
prof["LGUs"] = L.groupby("cluster").apply(lambda d: ", ".join(d.index)); prof

Chosen: KM-C hazard log1p k = 3 silhouette = 0.452 sizes = [8, 3, 3]


bootstrap ARI mean = 0.826 (≥0.75 stable, 0.6–0.75 fair, <0.6 unstable)


,flood_100yr_high_pct,flood_100yr_medium_pct,flood_100yr_low_pct,ss_ssa4_any_pct,flood_100yr_any_pct,LGUs
cluster,,,,,,
C1,7.05,7.07,3.49,7.90,17.61,"Cabangan, Candelaria, Masinloc, Olongapo, Pala..."
C2,12.35,8.62,7.69,0.91,28.66,"Botolan, Castillejos, San Marcelino"
C3,14.77,24.32,8.38,14.77,47.47,"Iba, San Felipe, San Narciso"


In [10]:
m = bounds.merge(L[["cluster"]], left_on="lgu_name", right_index=True)
fig, ax = plt.subplots(figsize=(5.5, 8))
cl = sorted(m.cluster.unique())
for i, c in enumerate(cl): m[m.cluster == c].plot(ax=ax, color=CAT[i], edgecolor="white", lw=.8, label=c)
for _, r in m.iterrows():
    p = r.geometry.representative_point(); ax.annotate(r.lgu_name, (p.x, p.y), fontsize=7, ha="center")
from matplotlib.patches import Patch
import textwrap
ax.legend(handles=[Patch(color=CAT[i], label=textwrap.fill(f"{c}: " + prof.loc[c, 'LGUs'], 38)) for i, c in enumerate(cl)],
          frameon=False, fontsize=7.5, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.set_axis_off(); ax.set_title(f"k-Means LGU exposure clusters ({best_name}, k={best_k})"); savefig("p2_03_cluster_map")

### 2.3 Association rules (mlxtend)
**(a) Daily rainfall regime → next-day extreme rain.** Each transaction is one day in one distinct POWER cell (≈50,000 transactions), so the support/confidence/lift values are stable.

**(b) LGU hazard tier ↔ rainfall regime.** This is the version the brief asked for, but with only 14 transactions. ⚠️ A rule with support 3/14 is three towns. These rules are reported for completeness and flagged unreliable.

In [11]:
# ---- Daily table used by 2.3a and section 3 ----
D = rain[rain.lgu_name.isin(REPS)].sort_values(["lgu_name", "date"]).reset_index(drop=True)
D["cell"] = D.lgu_name
base = D[(D.date.dt.year >= BASELINE_YEARS[0]) & (D.date.dt.year <= BASELINE_YEARS[1])]
THR = base[base.PRECTOTCORR >= 1].groupby("cell").PRECTOTCORR.quantile(.95)   # extreme threshold per cell, 1981–2010
print("Extreme-day thresholds (mm, 95th pct of 1981–2010 wet days):", THR.round(1).to_dict())
g = D.groupby("cell")
D["extreme_today"] = (D.PRECTOTCORR >= D.cell.map(THR)).astype(int)
D["y_next_extreme"] = g.extreme_today.shift(-1)
D["rain_next3_mm"] = g.PRECTOTCORR.transform(lambda s: s[::-1].rolling(3, min_periods=3).sum()[::-1].shift(-1))
D["rh_change"] = g.RH2M.diff()
D["rain_lag1"] = g.PRECTOTCORR.shift(1)
D = D.dropna(subset=["y_next_extreme", "rain_next3_mm", "rain_30d_mm", "rh_change", "rain_lag1"]).reset_index(drop=True)
D["y_next_extreme"] = D.y_next_extreme.astype(int)
D["year"] = D.date.dt.year
print("daily modelling rows:", len(D), "| next-day extreme rate:", round(D.y_next_extreme.mean()*100, 2), "%")

def band(s, edges, labels): return pd.cut(s, edges, labels=labels, right=False).astype(str)
T = pd.DataFrame({
    "season=wet": D.season.eq("wet"), "peak=Jul-Sep": D.month.isin([7, 8, 9]),
    **{f"rain_today={b}": band(D.PRECTOTCORR, [0, 1, 10, 30, 1e9], ["<1", "1-10", "10-30", ">=30"]).eq(b)
       for b in ["<1", "1-10", "10-30", ">=30"]},
    **{f"rain_7d={b}": band(D.rain_7d_mm, [0, 20, 100, 200, 1e9], ["<20", "20-100", "100-200", ">=200"]).eq(b)
       for b in ["<20", "20-100", "100-200", ">=200"]},
    "RH>=88%": D.RH2M >= 88, "extreme_today": D.extreme_today.eq(1),
    "NEXT_DAY_EXTREME": D.y_next_extreme.eq(1)})
fi = apriori(T, min_support=0.002, use_colnames=True, max_len=4)
rules = association_rules(fi, metric="confidence", min_threshold=0.05)
rules = rules[rules.consequents.apply(lambda c: c == frozenset({"NEXT_DAY_EXTREME"}))].copy()
rules["antecedents"] = rules.antecedents.apply(lambda s: " & ".join(sorted(s)))
top_rules = rules.sort_values(["lift", "support"], ascending=False)[["antecedents", "support", "confidence", "lift"]].head(10)
base_rate = D.y_next_extreme.mean()
log_run("AR1", "descriptive", "rainfall regime → next-day extreme", "Apriori + association rules",
        list(T.columns[:-1]), {"min_support": 0.002, "min_confidence": 0.05, "max_len": 4},
        f"all {len(T)} cell-days", {"rules_found": len(rules), "best_lift": top_rules.lift.max(),
                                     "best_confidence": top_rules.confidence.iloc[0], "base_rate": base_rate},
        "first run")
print(f"base rate P(next-day extreme) = {base_rate:.3f}"); top_rules.round(3)

Extreme-day thresholds (mm, 95th pct of 1981–2010 wet days): {'Botolan': 53.7, 'Candelaria': 40.2, 'Santa Cruz': 44.7}
daily modelling rows: 50130 | next-day extreme rate: 2.84 %
base rate P(next-day extreme) = 0.028


,antecedents,support,confidence,lift
1608,extreme_today & peak=Jul-Sep & rain_7d=>=200,0.006,0.454,15.998
1621,RH>=88% & extreme_today & peak=Jul-Sep,0.009,0.449,15.815
1750,RH>=88% & extreme_today & rain_7d=>=200,0.006,0.446,15.720
465,extreme_today & peak=Jul-Sep,0.009,0.437,15.409
966,extreme_today & peak=Jul-Sep & season=wet,0.009,0.437,15.409
1526,extreme_today & peak=Jul-Sep & rain_today=>=30,0.009,0.437,15.409
1284,extreme_today & rain_7d=>=200 & season=wet,0.006,0.429,15.108
619,extreme_today & rain_7d=>=200,0.006,0.427,15.067
1709,extreme_today & rain_7d=>=200 & rain_today=>=30,0.006,0.427,15.067
1297,RH>=88% & extreme_today & season=wet,0.010,0.422,14.866


In [12]:
# ---- 2.3b: LGU-level rules (n = 14) ----
tier = pd.qcut(L.flood_100yr_high_pct.rank(method="first"), 3, labels=["low", "mid", "high"])
TL = pd.DataFrame({
    **{f"flood_tier={t}": tier.eq(t) for t in ["low", "mid", "high"]},
    **{f"rain_cell={c}": L.rain_cell.eq(c) for c in REPS},
    "surge_SSA4>=1%": L.ss_ssa4_any_pct >= 1, "mapped_5_25yr": L.mapped_5_25yr})
fiL = apriori(TL, min_support=2/14, use_colnames=True, max_len=3)
rL = association_rules(fiL, metric="confidence", min_threshold=.4)
rL = rL[rL.consequents.apply(lambda c: len(c) == 1 and next(iter(c)).startswith("flood_tier"))].copy()
for c in ["antecedents", "consequents"]: rL[c] = rL[c].apply(lambda s: " & ".join(sorted(s)))
log_run("AR2", "descriptive", "LGU tier ↔ rain/surge regime", "Apriori + association rules", list(TL.columns),
        {"min_support": "2/14", "min_confidence": 0.4, "max_len": 3}, "all 14 LGUs",
        {"rules_found": len(rL), "best_lift": rL.lift.max() if len(rL) else np.nan},
        "brief's requested form", "unreliable: 14 transactions; support of 2/14 = two towns")
rL.sort_values("lift", ascending=False)[["antecedents", "consequents", "support", "confidence", "lift"]].head(8).round(3)

,antecedents,consequents,support,confidence,lift
0,rain_cell=Candelaria,flood_tier=low,0.143,0.500,1.400
9,mapped_5_25yr,flood_tier=high,0.357,0.500,1.400
43,mapped_5_25yr & surge_SSA4>=1%,flood_tier=high,0.286,0.500,1.400
21,rain_cell=Candelaria & surge_SSA4>=1%,flood_tier=low,0.143,0.500,1.400
38,mapped_5_25yr & rain_cell=Botolan,flood_tier=high,0.286,0.444,1.244
6,rain_cell=Botolan,flood_tier=high,0.286,0.444,1.244
34,rain_cell=Botolan & surge_SSA4>=1%,flood_tier=high,0.214,0.429,1.200
1,surge_SSA4>=1%,flood_tier=low,0.357,0.417,1.167


### 2.4 Similarity network (networkx)
Nodes are LGUs. An edge joins two LGUs whose standardised hazard profiles (the chosen clustering features) have cosine similarity ≥ 0.5, weighted by that similarity. Communities come from greedy modularity, and they are compared against the k-Means clusters and the rainfall cells. The node layout uses real coordinates, so geography stays readable.

In [13]:
Xs = best["Xs"]; nrm = Xs/np.linalg.norm(Xs, axis=1, keepdims=True); S = nrm @ nrm.T
G = nx.Graph(); names_ = L.index.tolist()
for i, n in enumerate(names_): G.add_node(n, rain_cell=L.rain_cell.iloc[i], cluster=L.cluster.iloc[i])
for i in range(len(names_)):
    for j in range(i+1, len(names_)):
        if S[i, j] >= .5: G.add_edge(names_[i], names_[j], weight=float(S[i, j]))
comms = list(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"))
comm_of = {n: k for k, c in enumerate(comms) for n in c}
Q = nx.algorithms.community.modularity(G, comms, weight="weight")
ari_c_k = adjusted_rand_score([comm_of[n] for n in names_], L.cluster)
ari_c_r = adjusted_rand_score([comm_of[n] for n in names_], L.rain_cell)
cent = nx.degree_centrality(G)
log_run("NW1", "descriptive", "LGU hazard-similarity network", "networkx graph + greedy modularity", best["cols"],
        {"edge_rule": "cosine >= 0.5"}, "full data (n=14)",
        {"edges": G.number_of_edges(), "communities": len(comms), "modularity": Q, "ARI_vs_kmeans": ari_c_k,
         "ARI_vs_rain_cell": ari_c_r}, "first run", "exploratory (n=14)")
pos = {n: (L.loc[n, "centroid_lon"], L.loc[n, "centroid_lat"]) for n in names_}
plt.figure(figsize=(6, 8))
bounds.boundary.plot(ax=plt.gca(), color="#c9c8c3", lw=.6)
nx.draw_networkx_edges(G, pos, width=[3*d["weight"]**2 for *_, d in G.edges(data=True)], edge_color="#8a8984", alpha=.7)
nx.draw_networkx_nodes(G, pos, node_color=[CAT[comm_of[n] % len(CAT)] for n in names_],
                       node_size=[300 + 900*cent[n] for n in names_], edgecolors="white")
nx.draw_networkx_labels(G, pos, font_size=7)
plt.title(f"Hazard-profile similarity network ({len(comms)} communities, Q = {Q:.2f})"); plt.axis("off")
savefig("p2_04_similarity_network")
print("communities:", [sorted(c) for c in comms])

communities: [['Cabangan', 'Candelaria', 'Masinloc', 'Olongapo', 'Palauig', 'Santa Cruz', 'Subic'], ['Iba', 'San Felipe', 'San Narciso'], ['Castillejos', 'San Marcelino'], ['Botolan'], ['San Antonio']]


## 3. Predictive models — daily rainfall (main section)
**Task:** using only what is known on day *t*, predict whether day *t+1* will be an **extreme-rain day**. An extreme day means rain ≥ the cell's 95th-percentile wet-day amount, with that threshold computed on the 1981–2010 normal so it doesn't leak future data. This is DIA's planned rainfall-trigger layer. It is a *rainfall* forecast, not a flood forecast, because no observed flood records exist (Phase 1, §5).

**Validation:**
- **Forward-chaining CV.** Train on all years before each block, then test on 2001–05, 2006–10, 2011–15 and 2016–20.
- **Untouched hold-out:** 2021 to the latest day.

The decision threshold is tuned on training predictions only, to maximise F1. Positives are rare (~2–3% of days), so accuracy is misleading on its own. **PR-AUC, recall and precision are the headline metrics.**

### 3.1 Classification: next-day extreme rain

In [14]:
FS1 = ["PRECTOTCORR", "rain_lag1", "rain_3d_mm", "rain_7d_mm", "rain_30d_mm", "month_sin", "month_cos", "season_wet"]
FS2 = FS1 + ["T2M", "RH2M", "rh_change"]
for c in REPS: D[f"cell_{c.replace(' ', '_')}"] = (D.cell == c).astype(int)
CELL_COLS = [c for c in D if c.startswith("cell_")]
FS2c = FS2 + CELL_COLS
dev = D[D.year < HOLDOUT_START].reset_index(drop=True); hold = D[D.year >= HOLDOUT_START].reset_index(drop=True)
folds = [(np.where(dev.year < a)[0], np.where((dev.year >= a) & (dev.year <= b))[0]) for a, b in CV_TEST_BLOCKS]
print("dev rows", len(dev), "| hold-out rows", len(hold), f"({hold.date.min().date()} → {hold.date.max().date()})")

def tuned_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p); f = 2*pr*rc/np.clip(pr+rc, 1e-9, None)
    return float(th[np.nanargmax(f[:-1])]) if len(th) else .5

def clf_metrics(y, p, thr):
    yh = (p >= thr).astype(int)
    return {"PR_AUC": average_precision_score(y, p), "ROC_AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p),
            "precision": precision_score(y, yh, zero_division=0), "recall": recall_score(y, yh), "F1": f1_score(y, yh),
            "accuracy": accuracy_score(y, yh), "support_pos": int(y.sum()), "support_neg": int((1-y).sum())}

def cv_eval(make, feats, df=dev):
    rows = []
    for tr, te in folds:
        mdl = make().fit(df.loc[tr, feats], df.loc[tr, "y_next_extreme"])
        ptr = mdl.predict_proba(df.loc[tr, feats])[:, 1]; pte = mdl.predict_proba(df.loc[te, feats])[:, 1]
        rows.append(clf_metrics(df.loc[te, "y_next_extreme"].to_numpy(), pte, tuned_threshold(df.loc[tr, "y_next_extreme"], ptr)))
    r = pd.DataFrame(rows); out = r.mean().to_dict(); out["PR_AUC_sd"] = r.PR_AUC.std()
    out["support_pos"] = int(r.support_pos.sum()); out["support_neg"] = int(r.support_neg.sum()); return out
CVDESC = "forward-chaining CV, 4 test blocks 2001–2020"

dev rows 43743 | hold-out rows 6387 (2021-01-01 → 2026-10-30)


In [15]:
# ---- Baselines ----
class Climatology:
    # P(next-day extreme | cell, month) estimated from training data
    def fit(self, X, y):
        self.t = pd.DataFrame({"m": X.month.values, "c": X.cell.values, "y": y.values}).groupby(["c", "m"]).y.mean(); self.g = y.mean(); return self
    def predict_proba(self, X):
        p = pd.Series(list(zip(X.cell, X.month))).map(self.t.to_dict()).fillna(self.g).to_numpy(); return np.c_[1-p, p]
class Persistence:
    def fit(self, X, y): return self
    def predict_proba(self, X): p = X.extreme_today.to_numpy(float); return np.c_[1-p, p]

R = {}
R["B0"] = cv_eval(Climatology, ["month", "cell"]); log_run("B0", "predictive-daily", "next-day extreme rain", "Climatology baseline (cell × month rate)", "cell, month", {}, CVDESC, R["B0"], "reference: no weather information")
R["B1"] = cv_eval(Persistence, ["extreme_today"]); log_run("B1", "predictive-daily", "next-day extreme rain", "Persistence baseline", "extreme_today", {}, CVDESC, R["B1"], "reference: tomorrow = today")
pd.DataFrame(R).T[["PR_AUC", "ROC_AUC", "precision", "recall", "F1"]].round(3)

,PR_AUC,ROC_AUC,precision,recall,F1
B0,0.105,0.826,0.110,0.485,0.179
B1,0.178,0.689,0.398,0.398,0.398


In [16]:
# ---- Round 1: four model families on FS1 (rain + calendar) ----
MODELS_R1 = {
  "LR1":  ("Logistic regression", lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000)), {"C": 1.0, "class_weight": "balanced"}),
  "LDA1": ("Linear Discriminant Analysis", lambda: make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")), {"solver": "lsqr", "shrinkage": "auto"}),
  "DT1":  ("Decision tree", lambda: DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, class_weight="balanced", random_state=SEED), {"max_depth": 4, "min_samples_leaf": 100, "class_weight": "balanced"}),
  "MLP1": ("MLP neural network", lambda: make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16,), alpha=1e-3, early_stopping=True, max_iter=300, random_state=SEED)), {"hidden": [16], "alpha": 1e-3, "early_stopping": True}),
}
for rid, (alg, mk, prm) in MODELS_R1.items():
    R[rid] = cv_eval(mk, FS1); log_run(rid, "predictive-daily", "next-day extreme rain", alg, FS1, prm, CVDESC, R[rid], "round 1: rain + calendar features")
pd.DataFrame(R).T[["PR_AUC", "PR_AUC_sd", "ROC_AUC", "precision", "recall", "F1", "Brier"]].round(3)

,PR_AUC,PR_AUC_sd,ROC_AUC,precision,recall,F1,Brier
B0,0.105,0.030,0.826,0.110,0.485,0.179,0.030
B1,0.178,0.024,0.689,0.398,0.398,0.398,0.038
LR1,0.378,0.069,0.931,0.416,0.439,0.427,0.106
LDA1,0.372,0.071,0.932,0.389,0.477,0.428,0.036
DT1,0.293,0.043,0.922,0.350,0.530,0.422,0.120
MLP1,0.233,0.134,0.800,0.236,0.458,0.283,0.030


In [17]:
# ---- Round 2: add humidity/temperature (FS2) and cell dummies (FS2c) ----
for rid, fs, note in [("LR2", FS2, "round 2: + T2M, RH2M, RH change"), ("LR3", FS2c, "round 2b: + cell dummies")]:
    R[rid] = cv_eval(MODELS_R1["LR1"][1], fs); log_run(rid, "predictive-daily", "next-day extreme rain", "Logistic regression", fs, MODELS_R1["LR1"][2], CVDESC, R[rid], note)
for rid, fs, note in [("LDA2", FS2c, "round 2: FS2 + cell dummies")]:
    R[rid] = cv_eval(MODELS_R1["LDA1"][1], fs); log_run(rid, "predictive-daily", "next-day extreme rain", "Linear Discriminant Analysis", fs, MODELS_R1["LDA1"][2], CVDESC, R[rid], note)
pd.DataFrame({k: R[k] for k in ["LR1", "LR2", "LR3", "LDA1", "LDA2"]}).T[["PR_AUC", "ROC_AUC", "precision", "recall", "F1"]].round(3)

,PR_AUC,ROC_AUC,precision,recall,F1
LR1,0.378,0.931,0.416,0.439,0.427
LR2,0.393,0.935,0.413,0.504,0.453
LR3,0.394,0.935,0.398,0.516,0.449
LDA1,0.372,0.932,0.389,0.477,0.428
LDA2,0.372,0.932,0.397,0.457,0.424


In [18]:
# ---- Round 3: hyper-parameter search (trees, MLP, LR regularisation) on FS2c ----
grid = []
for C_ in [0.01, 0.1, 1.0]:
    grid.append((f"LR-C{C_}", "Logistic regression", lambda C_=C_: make_pipeline(StandardScaler(), LogisticRegression(C=C_, class_weight="balanced", max_iter=2000)), {"C": C_, "class_weight": "balanced"}))
for d in [3, 5, 8]:
    for leaf in [50, 200]:
        grid.append((f"DT-d{d}-l{leaf}", "Decision tree", lambda d=d, leaf=leaf: DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf, class_weight="balanced", random_state=SEED), {"max_depth": d, "min_samples_leaf": leaf, "class_weight": "balanced"}))
for h in [(16,), (32, 16)]:
    for a in [1e-4, 1e-2]:
        grid.append((f"MLP-{'x'.join(map(str, h))}-a{a}", "MLP neural network", lambda h=h, a=a: make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=h, alpha=a, early_stopping=True, max_iter=400, random_state=SEED)), {"hidden": list(h), "alpha": a, "early_stopping": True}))
prev = {"Logistic regression": "LR3", "Decision tree": "DT1", "MLP neural network": "MLP1"}
for rid, alg, mk, prm in grid:
    R[rid] = cv_eval(mk, FS2c)
    delta = R[rid]["PR_AUC"] - R[prev[alg]]["PR_AUC"]
    log_run(rid, "predictive-daily", "next-day extreme rain", alg, FS2c, prm, CVDESC, R[rid],
            f"round 3 tuning; ΔPR-AUC vs {prev[alg]} = {delta:+.3f}")
res = pd.DataFrame(R).T; res[["PR_AUC", "PR_AUC_sd", "ROC_AUC", "precision", "recall", "F1"]].sort_values("PR_AUC", ascending=False).round(3).head(12)

,PR_AUC,PR_AUC_sd,ROC_AUC,precision,recall,F1
LR-C1.0,0.394,0.062,0.935,0.398,0.516,0.449
LR3,0.394,0.062,0.935,0.398,0.516,0.449
LR-C0.1,0.394,0.062,0.935,0.407,0.514,0.454
LR2,0.393,0.063,0.935,0.413,0.504,0.453
LR-C0.01,0.391,0.062,0.935,0.402,0.506,0.447
LR1,0.378,0.069,0.931,0.416,0.439,0.427
MLP-32x16-a0.01,0.374,0.073,0.928,0.365,0.521,0.423
MLP-32x16-a0.0001,0.373,0.064,0.928,0.417,0.458,0.426
LDA2,0.372,0.070,0.932,0.397,0.457,0.424
LDA1,0.372,0.071,0.932,0.389,0.477,0.428


In [19]:
# ---- Select per family (best CV PR-AUC), then refit on 1981–2020 and score the untouched hold-out ----
fam = {}
FEATS_OF = {k: FS1 for k in MODELS_R1} | {"LR2": FS2, "LR3": FS2c, "LDA2": FS2c} | {g[0]: FS2c for g in grid}
MAKER_OF = {k: v[1] for k, v in MODELS_R1.items()} | {"LR2": MODELS_R1["LR1"][1], "LR3": MODELS_R1["LR1"][1], "LDA2": MODELS_R1["LDA1"][1]} | {g[0]: g[2] for g in grid}
ALG_OF = {k: v[0] for k, v in MODELS_R1.items()} | {"LR2": "Logistic regression", "LR3": "Logistic regression", "LDA2": "Linear Discriminant Analysis"} | {g[0]: g[1] for g in grid}
for rid in MAKER_OF:
    a = ALG_OF[rid]
    if a not in fam or R[rid]["PR_AUC"] > R[fam[a]]["PR_AUC"]: fam[a] = rid
print("best per family:", fam)

FINAL, HOLD = {}, {}
for a, rid in fam.items():
    f = FEATS_OF[rid]; mdl = MAKER_OF[rid]().fit(dev[f], dev.y_next_extreme)
    thr = tuned_threshold(dev.y_next_extreme, mdl.predict_proba(dev[f])[:, 1])
    p = mdl.predict_proba(hold[f])[:, 1]
    HOLD[a] = clf_metrics(hold.y_next_extreme.to_numpy(), p, thr) | {"threshold": thr}
    FINAL[a] = {"model": mdl, "features": f, "threshold": thr, "run_id": rid, "p_hold": p}
    log_run(f"H-{rid}", "predictive-daily", "next-day extreme rain", a, f, {"from_run": rid, "threshold": round(thr, 4)},
            f"hold-out {HOLDOUT_START}–{hold.year.max()} (trained 1981–{HOLDOUT_START-1})", HOLD[a], "final test of best config")
for nm, mk, f in [("Climatology baseline", Climatology, ["month", "cell"]), ("Persistence baseline", Persistence, ["extreme_today"])]:
    mdl = mk().fit(dev[f], dev.y_next_extreme); p = mdl.predict_proba(hold[f])[:, 1]
    HOLD[nm] = clf_metrics(hold.y_next_extreme.to_numpy(), p, tuned_threshold(dev.y_next_extreme, mdl.predict_proba(dev[f])[:, 1]))
    log_run(f"H-{nm[:4]}", "predictive-daily", "next-day extreme rain", nm, f, {}, f"hold-out {HOLDOUT_START}–{hold.year.max()}", HOLD[nm], "reference")
holdtab = pd.DataFrame(HOLD).T[["PR_AUC", "ROC_AUC", "precision", "recall", "F1", "accuracy", "Brier", "support_pos", "support_neg"]]
holdtab.round(3)

best per family: {'Logistic regression': 'LR3', 'Linear Discriminant Analysis': 'LDA2', 'Decision tree': 'DT-d8-l50', 'MLP neural network': 'MLP-32x16-a0.01'}


,PR_AUC,ROC_AUC,precision,recall,F1,accuracy,Brier,support_pos,support_neg
Logistic regression,0.350,0.917,0.407,0.511,0.453,0.957,0.109,223.0,6164.0
Linear Discriminant Analysis,0.334,0.910,0.371,0.511,0.430,0.953,0.038,223.0,6164.0
Decision tree,0.351,0.881,0.368,0.493,0.421,0.953,0.106,223.0,6164.0
MLP neural network,0.401,0.916,0.404,0.489,0.442,0.957,0.026,223.0,6164.0
Climatology baseline,0.106,0.823,0.120,0.502,0.194,0.855,0.032,223.0,6164.0
Persistence baseline,0.198,0.700,0.422,0.422,0.422,0.960,0.040,223.0,6164.0


In [20]:
# Final model = best mean CV PR-AUC (chosen BEFORE looking at the hold-out; the hold-out only reports)
BEST = max(FINAL, key=lambda a: R[FINAL[a]["run_id"]]["PR_AUC"])
b = FINAL[BEST]; yh = (b["p_hold"] >= b["threshold"]).astype(int)
print(f"Selected by CV: {BEST} (run {b['run_id']}, CV PR-AUC {R[b['run_id']]['PR_AUC']:.3f}), threshold = {b['threshold']:.3f}")
print(f"Hold-out PR-AUC of each family: " + ", ".join(f"{a} {HOLD[a]['PR_AUC']:.3f}" for a in FINAL) + "\n")
print(classification_report(hold.y_next_extreme, yh, target_names=["not extreme", "extreme"], digits=3))
cm = confusion_matrix(hold.y_next_extreme, yh); print("confusion matrix [[TN FP],[FN TP]]:\n", cm)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for i, a in enumerate(FINAL):
    pr, rc, _ = precision_recall_curve(hold.y_next_extreme, FINAL[a]["p_hold"])
    ax[0].plot(rc, pr, lw=2, color=CAT[i], label=f"{a} (AP={HOLD[a]['PR_AUC']:.2f})")
ax[0].axhline(hold.y_next_extreme.mean(), color="#52514e", ls="--", lw=1, label="no-skill rate")
ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].set_title("Hold-out precision–recall"); ax[0].legend(frameon=False, fontsize=8)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[1], xticklabels=["pred no", "pred extreme"], yticklabels=["no", "extreme"])
ax[1].grid(False); ax[1].set_title(f"{BEST}: hold-out confusion matrix"); savefig("p2_05_daily_holdout")

pi = permutation_importance(b["model"], hold[b["features"]], hold.y_next_extreme, scoring="average_precision", n_repeats=10, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=b["features"]).sort_values()
plt.figure(figsize=(6, 4)); plt.barh(imp.index, imp.values, color=CAT[0]); plt.xlabel("drop in hold-out PR-AUC when shuffled")
plt.title(f"Permutation importance — {BEST}"); savefig("p2_06_daily_importance")
if "Decision tree" in FINAL:
    print(export_text(FINAL["Decision tree"]["model"], feature_names=FINAL["Decision tree"]["features"], max_depth=3))

Selected by CV: Logistic regression (run LR3, CV PR-AUC 0.394), threshold = 0.909
Hold-out PR-AUC of each family: Logistic regression 0.350, Linear Discriminant Analysis 0.334, Decision tree 0.351, MLP neural network 0.401

              precision    recall  f1-score   support

 not extreme      0.982     0.973     0.978      6164
     extreme      0.407     0.511     0.453       223

    accuracy                          0.957      6387
   macro avg      0.695     0.742     0.715      6387
weighted avg      0.962     0.957     0.959      6387

confusion matrix [[TN FP],[FN TP]]:
 [[5998  166]
 [ 109  114]]


|--- PRECTOTCORR <= 10.49
|   |--- PRECTOTCORR <= 3.30
|   |   |--- month_sin <= -0.25
|   |   |   |--- T2M <= 25.21
|   |   |   |   |--- truncated branch of depth 4
|   |   |   |--- T2M >  25.21
|   |   |   |   |--- truncated branch of depth 5
|   |   |--- month_sin >  -0.25
|   |   |   |--- rain_3d_mm <= 51.07
|   |   |   |   |--- truncated branch of depth 4
|   |   |   |--- rain_3d_mm >  51.07
|   |   |   |   |--- class: 1
|   |--- PRECTOTCORR >  3.30
|   |   |--- RH2M <= 81.92
|   |   |   |--- PRECTOTCORR <= 8.15
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- PRECTOTCORR >  8.15
|   |   |   |   |--- truncated branch of depth 4
|   |   |--- RH2M >  81.92
|   |   |   |--- season_wet <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- season_wet >  0.50
|   |   |   |   |--- truncated branch of depth 5
|--- PRECTOTCORR >  10.49
|   |--- PRECTOTCORR <= 20.79
|   |   |--- month_sin <= 0.25
|   |   |   |--- rain_3d_mm <= 124.91
|   |   |   |   |---

In [21]:
# Sanity check on a known hold-out event: Aug 2026 habagat (Olongapo is in the Botolan cell)
cell_olo = rep_of["Olongapo"]
ev = hold[(hold.cell == cell_olo) & (hold.date.between("2026-08-24", "2026-08-31"))].copy()
# Scores from a class-balanced model are rankings, not calibrated probabilities: compare them with the tuned threshold.
ev["model_score"] = FINAL[BEST]["model"].predict_proba(ev[FINAL[BEST]["features"]])[:, 1].round(3)
ev["warn_for_tomorrow"] = ev.model_score >= FINAL[BEST]["threshold"]
ev["tomorrow_was_extreme"] = ev.y_next_extreme.astype(bool)
print(f"decision threshold = {FINAL[BEST]['threshold']:.3f}")
ev[["date", "PRECTOTCORR", "rain_3d_mm", "model_score", "warn_for_tomorrow", "tomorrow_was_extreme"]]

decision threshold = 0.909


,date,PRECTOTCORR,rain_3d_mm,model_score,warn_for_tomorrow,tomorrow_was_extreme
2061,2026-08-24,19.64,51.11,0.731,False,False
2062,2026-08-25,21.18,58.90,0.753,False,False
2063,2026-08-26,15.41,56.23,0.592,False,False
2064,2026-08-27,27.81,64.40,0.855,False,True
2065,2026-08-28,83.78,127.00,0.997,True,True
2066,2026-08-29,62.11,173.70,0.936,True,False
2067,2026-08-30,47.42,193.31,0.945,True,False
2068,2026-08-31,19.36,128.89,0.641,False,False


### 3.2 Linear regression — next-3-day rainfall total
The target is rainfall over days t+1 to t+3. It is modelled both as log(1 + mm), to damp the heavy tail, and as raw mm. The final model is the one with the lowest CV RMSE in millimetres, because millimetres are what DIA would show. The same forward-chaining CV and hold-out are used.

In [22]:
def reg_eval(make, feats, df_tr, df_te, target="log"):
    # target='log': fit on log1p(mm), predictions clipped to the training range before back-transform
    # target='raw': fit on mm directly, predictions clipped at 0
    ytr = df_tr.rain_next3_mm; yte = df_te.rain_next3_mm
    if target == "log":
        m_ = make().fit(df_tr[feats], np.log1p(ytr)); pl = np.clip(m_.predict(df_te[feats]), 0, np.log1p(ytr.max())); pmm = np.expm1(pl)
    else:
        m_ = make().fit(df_tr[feats], ytr); pmm = np.clip(m_.predict(df_te[feats]), 0, None); pl = np.log1p(pmm)
    yl = np.log1p(yte)
    return m_, {"R2_log": r2_score(yl, pl), "RMSE_log": mean_squared_error(yl, pl) ** .5,
                "RMSE_mm": mean_squared_error(yte, pmm) ** .5, "MAE_mm": np.mean(np.abs(yte - pmm)),
                "R2_mm": r2_score(yte, pmm), "n_test": len(df_te)}
class ClimMean:
    def fit(self, X, y): self.t = pd.Series(y.values).groupby([X.cell.values, X.month.values]).mean(); self.g = y.mean(); return self
    def predict(self, X): return pd.Series(list(zip(X.cell, X.month))).map(self.t.to_dict()).fillna(self.g).to_numpy()
OLS = lambda: make_pipeline(StandardScaler(), LinearRegression())
REG = {"RB0": ("Climatology mean (cell × month)", ClimMean, ["cell", "month"], {"target": "raw"}, "raw", "reference"),
       "RG1": ("Linear regression (OLS)", OLS, FS1, {"target": "log1p"}, "log", "round 1: rain + calendar, log target"),
       "RG2": ("Linear regression (OLS)", OLS, FS2c, {"target": "log1p"}, "log", "round 2: + T2M, RH2M, RH change, cell"),
       "RG3": ("Ridge regression", lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0)), FS2c, {"alpha": 10.0, "target": "log1p"}, "log", "round 3: L2 penalty"),
       "RG4": ("Linear regression (OLS)", OLS, FS2c, {"target": "raw mm"}, "raw", "round 4: raw-mm target (log model under-predicts large totals after back-transform)")}
REGRES = {}
for rid, (alg, mk, f, prm, tgt, note) in REG.items():
    cv = pd.DataFrame([reg_eval(mk, f, dev.loc[tr], dev.loc[te], tgt)[1] for tr, te in folds]).mean().to_dict()
    REGRES[rid] = cv; log_run(rid, "predictive-daily", "next-3-day rainfall total (mm)", alg, f, prm, CVDESC, cv, note)
bestreg = min([k for k in REG if k != "RB0"], key=lambda k: REGRES[k]["RMSE_mm"])   # selected on CV RMSE in mm
reg_model, reg_hold = reg_eval(REG[bestreg][1], REG[bestreg][2], dev, hold, REG[bestreg][4])
_, reg_hold_b = reg_eval(ClimMean, ["cell", "month"], dev, hold, "raw")
log_run(f"H-{bestreg}", "predictive-daily", "next-3-day rainfall total (mm)", REG[bestreg][0], REG[bestreg][2], {"from_run": bestreg},
        f"hold-out {HOLDOUT_START}–{hold.year.max()}", reg_hold, "final test of best config")
log_run("H-RB0", "predictive-daily", "next-3-day rainfall total (mm)", "Climatology mean (cell × month)", "cell, month", {},
        f"hold-out {HOLDOUT_START}–{hold.year.max()}", reg_hold_b, "reference")
pd.DataFrame(REGRES | {f"HOLD {bestreg}": reg_hold, "HOLD RB0": reg_hold_b}).T.round(3)

,R2_log,RMSE_log,RMSE_mm,MAE_mm,R2_mm,n_test
RB0,0.552,1.036,34.865,16.776,0.272,5478.75
RG1,0.629,0.942,41.059,16.425,-0.005,5478.75
RG2,0.666,0.894,37.422,15.388,0.165,5478.75
RG3,0.666,0.894,37.422,15.388,0.165,5478.75
RG4,0.565,1.021,31.858,15.420,0.393,5478.75
HOLD RG4,0.541,0.995,31.870,15.945,0.392,6387.00
HOLD RB0,0.525,1.012,34.810,17.539,0.274,6387.00


## 4. Predictive models — LGU level (exploratory, n = 14)
⚠️ **Read these results as feasibility checks, not findings.**
- There are 14 rows. Rainfall features take only **3 distinct values** (2 among the 10 LGUs mapped at 5/25-yr).
- There are no external labels. The "risk tier" is defined from the 100-yr high-hazard share, so **no 100-yr flood feature is allowed as a predictor**, to avoid leakage.
- Validation uses leave-one-out CV. Each classifier is also run through a **permutation test** (200 label shuffles), which asks: is this better than chance?

In [23]:
# ---- 4.1 Linear regression: high-hazard growth (100yr − 5yr, pp) on the 10 mapped LGUs ----
M = L[L.mapped_5_25yr].copy(); loo = LeaveOneOut()
print("mapped LGUs:", len(M), "| distinct rainfall values among them:", M.rain_mean_annual_mm.nunique())
LG = {"LG1": (["rain_mean_annual_mm", "rain_mean_annual_max1d_mm"], "brief's spec: rainfall → growth"),
      "LG2": (["lgu_area_km2", "centroid_lat", "ss_ssa4_any_pct"], "swap to geographic/coastal features"),
      "LG3": (["flood_5yr_any_pct"], "baseline exposure only: do already-exposed LGUs grow more?")}
for rid, (f, note) in LG.items():
    pred = cross_val_predict(make_pipeline(StandardScaler(), LinearRegression()), M[f], M.high_growth_pp, cv=loo)
    base_pred = np.array([M.high_growth_pp.drop(i).mean() for i in M.index])
    met = {"R2_LOO": r2_score(M.high_growth_pp, pred), "RMSE_pp": mean_squared_error(M.high_growth_pp, pred) ** .5,
           "RMSE_mean_baseline_pp": mean_squared_error(M.high_growth_pp, base_pred) ** .5, "n": len(M)}
    rel = "unreliable: rainfall has 2 distinct values" if rid == "LG1" else "exploratory (n=10)"
    log_run(rid, "predictive-LGU", "high-hazard growth 5→100yr (pp)", "Linear regression (OLS)", f, {}, "leave-one-out (n=10)", met, note, rel)
    print(rid, {k: round(v, 3) for k, v in met.items()})

mapped LGUs: 10 | distinct rainfall values among them: 2
LG1 {'R2_LOO': -0.25, 'RMSE_pp': 2.244, 'RMSE_mean_baseline_pp': 2.23, 'n': 10}
LG2 {'R2_LOO': -0.896, 'RMSE_pp': 2.764, 'RMSE_mean_baseline_pp': 2.23, 'n': 10}
LG3 {'R2_LOO': -0.072, 'RMSE_pp': 2.078, 'RMSE_mean_baseline_pp': 2.23, 'n': 10}


In [24]:
# ---- 4.2 Risk-tier classification (terciles of corrected 100-yr high-hazard share) ----
y_t = tier.astype(str).to_numpy()
TF = ["rain_mean_annual_mm", "rain_mean_annual_max1d_mm", "ss_ssa4_any_pct", "lgu_area_km2", "centroid_lat", "centroid_lon"]
TIER_MODELS = {
  "TC1": ("Logistic regression", lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)), {"C": 1.0}),
  "TC2": ("Logistic regression", lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)), {"C": 0.1}),
  "TC3": ("Linear Discriminant Analysis", lambda: make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")), {"shrinkage": "auto"}),
  "TC4": ("Decision tree", lambda: DecisionTreeClassifier(max_depth=2, random_state=SEED), {"max_depth": 2}),
  "TC5": ("Decision tree", lambda: DecisionTreeClassifier(max_depth=3, random_state=SEED), {"max_depth": 3}),
  "TC6": ("MLP neural network", lambda: make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(8,), alpha=1e-2, max_iter=3000, random_state=SEED)), {"hidden": [8], "alpha": 1e-2}),
}
TIER_RES = {}
for rid, (alg, mk, prm) in TIER_MODELS.items():
    pred = cross_val_predict(mk(), L[TF], y_t, cv=loo)
    score, _, pval = permutation_test_score(mk(), L[TF], y_t, cv=loo, scoring="accuracy", n_permutations=200, random_state=SEED)
    met = {"accuracy_LOO": accuracy_score(y_t, pred), "precision_macro": precision_score(y_t, pred, average="macro", zero_division=0),
           "recall_macro": recall_score(y_t, pred, average="macro", zero_division=0), "perm_test_p": pval,
           "chance_accuracy": max(pd.Series(y_t).value_counts(normalize=True)), "support": "low/mid/high = " + "/".join(str((y_t == c).sum()) for c in ["low", "mid", "high"])}
    TIER_RES[rid] = met | {"pred": pred}
    log_run(rid, "predictive-LGU", "risk tier (terciles of 100yr high-hazard share)", alg, TF, prm, "leave-one-out + 200-permutation test", met,
            "first family run" if rid in ("TC1", "TC3", "TC4", "TC6") else "tuned complexity", "exploratory (n=14)" if pval < .05 else "unreliable: not better than chance (perm p ≥ 0.05)")
pd.DataFrame({k: {m: v for m, v in d.items() if m != "pred"} for k, d in TIER_RES.items()}).T

,accuracy_LOO,precision_macro,recall_macro,perm_test_p,chance_accuracy,support
TC1,0.285714,0.289683,0.283333,0.477612,0.357143,low/mid/high = 5/4/5
TC2,0.285714,0.216667,0.266667,0.363184,0.357143,low/mid/high = 5/4/5
TC3,0.357143,0.395238,0.35,0.393035,0.357143,low/mid/high = 5/4/5
TC4,0.428571,0.375,0.4,0.298507,0.357143,low/mid/high = 5/4/5
TC5,0.357143,0.345238,0.333333,0.378109,0.357143,low/mid/high = 5/4/5
TC6,0.285714,0.228571,0.266667,0.552239,0.357143,low/mid/high = 5/4/5


In [25]:
bt = max(TIER_RES, key=lambda k: (TIER_RES[k]["accuracy_LOO"], -TIER_RES[k]["perm_test_p"]))
print("Best tier model:", bt, TIER_MODELS[bt][0])
print(classification_report(y_t, TIER_RES[bt]["pred"], labels=["low", "mid", "high"], zero_division=0))
tier_model = TIER_MODELS[bt][1]().fit(L[TF], y_t)
pd.DataFrame({"true tier": y_t, "LOO predicted": TIER_RES[bt]["pred"], "high_100yr_pct": L.flood_100yr_high_pct.round(2)}, index=L.index)

Best tier model: TC4 Decision tree
              precision    recall  f1-score   support

         low       0.38      0.60      0.46         5
         mid       0.00      0.00      0.00         4
        high       0.75      0.60      0.67         5

    accuracy                           0.43        14
   macro avg       0.38      0.40      0.38        14
weighted avg       0.40      0.43      0.40        14



,true tier,LOO predicted,high_100yr_pct
lgu_name,,,
Botolan,high,low,15.97
Cabangan,mid,low,8.83
Candelaria,low,low,5.72
Castillejos,high,high,13.14
Iba,high,low,15.51
Masinloc,mid,low,7.83
Olongapo,mid,high,7.68
Palauig,low,low,6.11
San Antonio,low,mid,6.14


## 5. Hypothesis tests H1–H4 (re-run on corrected data, with bootstrap CIs)

In [26]:
B = 5000; H = {}
def boot_ci(fn, *arrays, B=B):
    n = len(arrays[0]); vals = []
    for _ in range(B):
        i = rng.integers(0, n, n); vals.append(fn(*[np.asarray(a)[i] for a in arrays]))
    return np.nanpercentile(vals, [2.5, 97.5])

# H1 — 100-yr vs 5-yr high-hazard share, paired; all 14 and mapped 10
for tag, df in [("all_14", L), ("mapped_10", L[L.mapped_5_25yr])]:
    a, b2 = df.flood_5yr_high_pct.to_numpy(), df.flood_100yr_high_pct.to_numpy()
    wres = stats.wilcoxon(b2, a, alternative="greater")
    d = b2 - a; n = len(d); rbc = (np.sum(stats.rankdata(np.abs(d))[d > 0]) - np.sum(stats.rankdata(np.abs(d))[d < 0])) / (n*(n+1)/2)
    H[f"H1_{tag}"] = {"n": n, "median_5yr": np.median(a), "median_100yr": np.median(b2), "median_diff_pp": np.median(d),
                      "median_diff_CI95": boot_ci(lambda x: np.median(x), d).round(2).tolist(),
                      "W": wres.statistic, "p_one_sided": wres.pvalue, "rank_biserial": rbc}
# H2 — wet vs dry season monthly totals, regional mean of distinct cells
reg_s = rain[rain.lgu_name.isin(REPS)].pivot(index="date", columns="lgu_name", values="PRECTOTCORR").mean(axis=1)
mon = reg_s.resample("MS").sum(min_count=25).dropna(); wet = mon[mon.index.month.isin(WET_SEASON)]; dry = mon[~mon.index.month.isin(WET_SEASON)]
u = stats.mannwhitneyu(wet, dry, alternative="greater")
H["H2"] = {"n_wet_months": len(wet), "n_dry_months": len(dry), "median_wet_mm": wet.median(), "median_dry_mm": dry.median(),
           "U": u.statistic, "p_one_sided": u.pvalue, "rank_biserial": 2*u.statistic/(len(wet)*len(dry)) - 1}
# H3 — Mann–Kendall (Kendall's tau vs year) + Sen's slope with 95% CI, per distinct cell
H["H3"] = {}
for c in REPS:
    s = rain[rain.lgu_name == c].set_index("date").PRECTOTCORR; y = s.groupby(s.index.year)
    am = y.max()[y.count() >= 360]; tau = stats.kendalltau(am.index, am.values); sl = stats.theilslopes(am.values, am.index)
    H["H3"][c] = {"years": len(am), "tau": tau.statistic, "p_two_sided": tau.pvalue, "sen_mm_per_decade": sl[0]*10,
                  "sen_CI95_mm_per_decade": [sl[2]*10, sl[3]*10]}
# H4 — Spearman, corrected shares
rho = stats.spearmanr(L.flood_100yr_any_pct, L.ss_ssa4_any_pct)
H["H4"] = {"n": len(L), "rho": rho.statistic, "p_two_sided": rho.pvalue,
           "rho_CI95": boot_ci(lambda x, y: stats.spearmanr(x, y).statistic, L.flood_100yr_any_pct, L.ss_ssa4_any_pct).round(2).tolist()}
def clean(o):
    if isinstance(o, dict): return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, np.ndarray)): return [clean(v) for v in o]
    return float(f"{float(o):.6g}") if isinstance(o, (float, np.floating, np.integer)) else o
H = clean(H); print(json.dumps(H, indent=1))

{
 "H1_all_14": {
  "n": 14,
  "median_5yr": 2.88908,
  "median_100yr": 7.88618,
  "median_diff_pp": 6.70576,
  "median_diff_CI95": [
   5.69,
   7.83
  ],
  "W": 105.0,
  "p_one_sided": 6.10352e-05,
  "rank_biserial": 1.0
 },
 "H1_mapped_10": {
  "n": 10,
  "median_5yr": 3.31604,
  "median_100yr": 10.4385,
  "median_diff_pp": 6.58553,
  "median_diff_CI95": [
   4.63,
   8.45
  ],
  "W": 55.0,
  "p_one_sided": 0.000976562,
  "rank_biserial": 1.0
 },
 "H2": {
  "n_wet_months": 276,
  "n_dry_months": 274,
  "median_wet_mm": 354.305,
  "median_dry_mm": 21.5117,
  "U": 74454.0,
  "p_one_sided": 2.22046e-86,
  "rank_biserial": 0.969057
 },
 "H3": {
  "Botolan": {
   "years": 45,
   "tau": 0.319192,
   "p_two_sided": 0.00199341,
   "sen_mm_per_decade": 16.5871,
   "sen_CI95_mm_per_decade": [
    4.79333,
    27.6
   ]
  },
  "Candelaria": {
   "years": 45,
   "tau": 0.337374,
   "p_two_sided": 0.00108581,
   "sen_mm_per_decade": 13.706,
   "sen_CI95_mm_per_decade": [
    5.2,
    23.8158
   

In [27]:
h1a, h1m, h2, h4 = H["H1_all_14"], H["H1_mapped_10"], H["H2"], H["H4"]
h3 = H["H3"]
txt = f'''
**RQ1 / RQ2 → H1.** On the 10 LGUs mapped in both scenarios, the median high-hazard share rises from {h1m["median_5yr"]:.2f}% (5-yr) to {h1m["median_100yr"]:.2f}% (100-yr), a median increase of {h1m["median_diff_pp"]:.2f} pp (95% CI {h1m["median_diff_CI95"][0]}–{h1m["median_diff_CI95"][1]}). Wilcoxon W = {h1m["W"]:.0f}, one-sided p = {h1m["p_one_sided"]:.2g}, rank-biserial r = {h1m["rank_biserial"]:.2f}. **H1 supported.** The all-14 version (p = {h1a["p_one_sided"]:.2g}) overstates the effect, because 4 LGUs have no 5-yr map.

**RQ3 → H2.** Median wet-season month {h2["median_wet_mm"]:.0f} mm vs dry-season {h2["median_dry_mm"]:.0f} mm; U = {h2["U"]:.0f}, p = {h2["p_one_sided"]:.2g}, rank-biserial r = {h2["rank_biserial"]:.2f}. **H2 supported**, a very large effect consistent with Type I climate.

**RQ4 → H3.** ''' + "; ".join(f'{c}: τ = {v["tau"]:.2f}, p = {v["p_two_sided"]:.3f}, Sen +{v["sen_mm_per_decade"]:.1f} mm/decade (95% CI {v["sen_CI95_mm_per_decade"][0]:.1f} to {v["sen_CI95_mm_per_decade"][1]:.1f})' for c, v in h3.items()) + f'''. **H3 supported (increasing)** in all three cells.

**RQ5 → H4.** Spearman ρ = {h4["rho"]:.2f} (95% CI {h4["rho_CI95"][0]} to {h4["rho_CI95"][1]}), p = {h4["p_two_sided"]:.2f}, n = {h4["n"]}. **H4 not supported.** The CI is wide and includes 0, so n = 14 cannot distinguish "no relation" from "moderate relation". DIA reports flood and storm surge as separate layers.
'''
from IPython.display import Markdown, display; display(Markdown(txt)); (OUT/"hypothesis_interpretation.md").write_text(txt)


**RQ1 / RQ2 → H1.** On the 10 LGUs mapped in both scenarios, the median high-hazard share rises from 3.32% (5-yr) to 10.44% (100-yr), a median increase of 6.59 pp (95% CI 4.63–8.45). Wilcoxon W = 55, one-sided p = 0.00098, rank-biserial r = 1.00. **H1 supported.** The all-14 version (p = 6.1e-05) overstates the effect, because 4 LGUs have no 5-yr map.

**RQ3 → H2.** Median wet-season month 354 mm vs dry-season 22 mm; U = 74454, p = 2.2e-86, rank-biserial r = 0.97. **H2 supported**, a very large effect consistent with Type I climate.

**RQ4 → H3.** Botolan: τ = 0.32, p = 0.002, Sen +16.6 mm/decade (95% CI 4.8 to 27.6); Candelaria: τ = 0.34, p = 0.001, Sen +13.7 mm/decade (95% CI 5.2 to 23.8); Santa Cruz: τ = 0.29, p = 0.005, Sen +14.3 mm/decade (95% CI 4.3 to 25.4). **H3 supported (increasing)** in all three cells.

**RQ5 → H4.** Spearman ρ = 0.26 (95% CI -0.39 to 0.79), p = 0.37, n = 14. **H4 not supported.** The CI is wide and includes 0, so n = 14 cannot distinguish "no relation" from "moderate relation". DIA reports flood and storm surge as separate layers.


1078

## 6. Text mining — not performed
No text corpus is in scope. NDRRMC situation reports exist only as PDFs and were not collected in Phase 1. Tokenising, stemming and n-gram extraction would have nothing real to run on, so this optional step is skipped rather than demonstrated on made-up text. It is recommended for a later phase, once incident reports are digitised (Phase 1 §5).

## 7. Results summary (run on the Phase 1 outputs of 2026-11-16; seeds fixed, so reruns reproduce these numbers)

**Descriptive**
- **Correlation:** the 100-yr low and medium shares track the 100-yr total (Spearman ρ 0.85–0.92), and the high share correlates 0.83 with it. Rainfall correlates −0.79 with latitude, but that only reflects the 3 POWER cells, not a physical north–south gradient.
- **k-Means:** the chosen solution uses hazard-only features with log1p, k = 3, silhouette 0.45, and a bootstrap ARI of 0.83, so it is stable. The clusters are:
  - **C1**, lower exposure (8 LGUs, including Olongapo and the north).
  - **C2**, inland river floodplains with almost no storm surge (Botolan, Castillejos, San Marcelino).
  - **C3**, coastal double exposure, high flood plus storm surge (Iba, San Felipe, San Narciso).
  - The rainfall-inclusive set (KM-A) mostly reproduced the 3 rainfall cells (ARI up to 0.80), so rainfall was dropped.
- **Association rules (daily):** "extreme today & Jul–Sep & 7-day rain ≥ 200 mm → extreme tomorrow" has confidence 0.45 and lift 16 against a 2.8% base rate. Extreme rain clusters in multi-day habagat spells. The LGU-level rules (n = 14) reach lift ≤ 1.4, which is not informative.
- **Network:** 5 communities (Q = 0.25) largely match the k-Means clusters (ARI 0.78). Botolan and San Antonio sit alone.

**Predictive, daily (validated)**
- **Next-day extreme rain.** All four model families beat both baselines. Forward-chaining CV PR-AUC: LR 0.39, MLP 0.37, LDA 0.37, Decision tree 0.33. The baselines scored 0.11 (climatology) and 0.18 (persistence).
- **Hold-out 2021–2026.** The final model is logistic regression, selected by CV. It scored PR-AUC 0.35, ROC-AUC 0.92, recall 0.51 and precision 0.41 (114 of 223 extreme days caught, 166 false alarms). The MLP scored highest on the hold-out (0.40), but it was not chosen on hold-out data, to avoid selection bias.
- **Next-3-day rainfall regression.** OLS on raw mm (RG4) gives hold-out RMSE 31.9 mm and R² 0.39, against 34.8 mm and 0.27 for climatology. The log-target models score better R² on the log scale but worse in mm after back-transform.

**Predictive, LGU level (exploratory)**
- **Growth regression (10 mapped LGUs):** every feature set has negative LOO R², no better than predicting the mean. Rainfall takes only 2 distinct values among these LGUs.
- **Risk-tier classifiers:** LOO accuracy is 0.29–0.43 against a chance rate of 0.36. No permutation test reached p < 0.05. The data cannot predict an LGU's flood tier from rainfall, storm surge or geography with n = 14, so tiers should come from the hazard shares themselves (the clusters above), not from a classifier.

## 8. Export: model artifacts and experiment log

In [28]:
artifacts = {
  "created_utc": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
  "daily_extreme_classifier": {"algorithm": BEST, "run_id": FINAL[BEST]["run_id"], "model": FINAL[BEST]["model"],
                               "features": FINAL[BEST]["features"], "decision_threshold": FINAL[BEST]["threshold"],
                               "extreme_thresholds_mm": THR.to_dict(), "baseline_years": BASELINE_YEARS,
                               "holdout_metrics": HOLD[BEST]},
  "daily_classifiers_all_families": {a: {k: v for k, v in d.items() if k != "p_hold"} for a, d in FINAL.items()},
  "next3day_rain_regressor": {"run_id": bestreg, "model": reg_model, "features": REG[bestreg][2], "target": REG[bestreg][4],
                              "holdout_metrics": reg_hold},
  "lgu_kmeans": {"set": best_name, "k": best_k, "features": best["cols"], "log1p": best["log"],
                 "scaler": StandardScaler().fit(np.log1p(L[best["cols"]]) if best["log"] else L[best["cols"]]),
                 "model": KMeans(n_clusters=best_k, n_init=50, random_state=SEED).fit(best["Xs"]),
                 "labels": L.cluster.to_dict(), "bootstrap_ARI": stab},
  "lgu_tier_classifier": {"run_id": bt, "model": tier_model, "features": TF, "classes": ["low", "mid", "high"],
                          "note": "exploratory, n=14", "loo_metrics": {k: v for k, v in TIER_RES[bt].items() if k != "pred"}},
  "hypothesis_tests": H,
}
joblib.dump(artifacts, OUT/"model_artifacts.pkl")
log = pd.DataFrame(LOG); log.to_csv(OUT/"experiment_log.csv", index=False)

def to_md(df):
    df = df.fillna("").astype(str).apply(lambda c: c.str.replace("|", "/", regex=False))
    return "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n" + "\n".join("| " + " | ".join(r) + " |" for r in df.values)
metric_cols = [c for c in log.columns if c not in ("run_id", "section", "task", "algorithm", "features", "hyperparameters", "validation", "change_vs_previous", "reliability")]
(OUT/"experiment_log.md").write_text("# DIA Phase 2 — Model Experiment Log\n\n" + to_md(log) + "\n")
print(len(log), "runs logged →", OUT/"experiment_log.csv")
shutil.make_archive("FSAI_DIA_Phase2_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("FSAI_DIA_Phase2_outputs.zip")
except Exception: pass
log[["run_id", "algorithm", "validation", "PR_AUC", "recall", "precision", "silhouette", "accuracy_LOO", "perm_test_p", "R2_log", "reliability"]].head(60)

64 runs logged → phase2_outputs/experiment_log.csv


,run_id,algorithm,validation,PR_AUC,recall,precision,silhouette,accuracy_LOO,perm_test_p,R2_log,reliability
0,D1,Spearman correlation matrix,full data (n=14),NaN,NaN,NaN,NaN,NaN,NaN,NaN,exploratory (n=14)
1,KM01,k-Means,full data (n=14),NaN,NaN,NaN,0.3981,NaN,NaN,NaN,exploratory (n=14)
2,KM02,k-Means,full data (n=14),NaN,NaN,NaN,0.4214,NaN,NaN,NaN,exploratory (n=14)
3,KM03,k-Means,full data (n=14),NaN,NaN,NaN,0.3531,NaN,NaN,NaN,exploratory (n=14)
4,KM04,k-Means,full data (n=14),NaN,NaN,NaN,0.3957,NaN,NaN,NaN,exploratory (n=14)
5,KM05,k-Means,full data (n=14),NaN,NaN,NaN,0.3491,NaN,NaN,NaN,exploratory (n=14)
6,KM06,k-Means,full data (n=14),NaN,NaN,NaN,0.4385,NaN,NaN,NaN,exploratory (n=14)
7,KM07,k-Means,full data (n=14),NaN,NaN,NaN,0.4035,NaN,NaN,NaN,exploratory (n=14)
8,KM08,k-Means,full data (n=14),NaN,NaN,NaN,0.4389,NaN,NaN,NaN,exploratory (n=14)
9,KM09,k-Means,full data (n=14),NaN,NaN,NaN,0.3991,NaN,NaN,NaN,exploratory (n=14)
